# Titanic Prediction Model

In [50]:
# Importing Libraries
import pandas as pd
import numpy as np


In [51]:
train = pd.read_csv('/Users/wallace/Documents/Projects/titanic-end-to-end-model/data/train.csv')
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


int64/float64 are already data types useable by a model, str dtypes by other cols are to be converted to be used. 

In [52]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Looking into the cols, survived col is here with value 1 and 0. 1 being survived and 0 not surviving, this will be the goal of this prediction model as this col of 'Survived' would only be available in this training dataset.

In [53]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Age is missing 177 out of 891 times (~20%), Cabin is missing 687 times (~77%), Embarked missing just 2. Cabin is not that useful as it is entirely almost empty while age and embarked are to be filled. 

Patterns of survibaility in terms of different traits of the passengers. The traits would be classified by the cols: Sex, Pclass, Age.. 

In [54]:
train.groupby('Sex')['Survived'].mean()

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

Survival rate by sex: female ≈ 74%, male ≈ 19%.
Women were roughly 4x more likely to survive than men, consistent with 
the 'women and children first' evacuation policy.

In [55]:
train.groupby('Pclass')['Survived'].mean()

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

Survival rate by Pclass: 
* 1 ≈ 63% 
* 2 ≈ 47%
* 3 ≈ 24% 

Pclass is the social class of the passengers with 1 being the highest class and 3 being the lowest class of passengers. These numbers shows that those that are 1st class passengers have a more likely chance to have survived, while those in the 3rd class has a lower chance to have survived

In [56]:
train["AgeGroup"] = pd.cut(train["Age"], 
                           bins=[0, 12, 18, 60, 100], 
                           labels=["Child", "Teenager", "Adult","Senior"])
train.groupby('AgeGroup')['Survived'].mean()

AgeGroup
Child       0.579710
Teenager    0.428571
Adult       0.388788
Senior      0.227273
Name: Survived, dtype: float64

Categorizing the Age Group col into different labels with pd.cut with bins ranging from 0-80. With this, running .groupby shows that survival rate by AgeGroup: 
* Child ≈ 58%
* Teenager ≈ 43%
* Adult ≈ 39%
* Senior ≈ 23%

As stated before with the policy "Women and Children First" shows that children and teenagers has more rates of survival. Suprisingly seniors have the lowest rate of survival. 

In [57]:
train["AgeGroup"].value_counts()

AgeGroup
Adult       553
Teenager     70
Child        69
Senior       22
Name: count, dtype: int64

From the counts of each passenger seniors have a relative low sample size, thus interpreting the survival rate with only 23% should be done with caution. 

In [58]:
# Fixing missing values
train["Age"] = train["Age"].fillna(train.groupby("Pclass")["Age"].transform("median"))
train["Embarked"] = train["Embarked"].fillna(train["Embarked"].mode()[0])
train = train.drop("Cabin", axis=1)

Re-run the age group categorization code cell to fill the missing values for the AgeGroup col.

In [59]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Embarked         0
AgeGroup       177
dtype: int64

In [60]:
train = train.drop(["PassengerId", "Name", "Ticket"], axis=1)
train.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'AgeGroup'],
      dtype='str')

In [61]:
# Turning categorical variables into numerical variables
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
train = pd.get_dummies(train, columns=['Embarked'], drop_first=True)
train = pd.get_dummies(train, columns=['AgeGroup'], drop_first=True)
train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S,AgeGroup_Teenager,AgeGroup_Adult,AgeGroup_Senior
0,0,3,0,22.0,1,0,7.2500,False,True,False,True,False
1,1,1,1,38.0,1,0,71.2833,False,False,False,True,False
2,1,3,1,26.0,0,0,7.9250,False,True,False,True,False
3,1,1,1,35.0,1,0,53.1000,False,True,False,True,False
4,0,3,0,35.0,0,0,8.0500,False,True,False,True,False


In [62]:
train.dtypes

Survived               int64
Pclass                 int64
Sex                    int64
Age                  float64
SibSp                  int64
Parch                  int64
Fare                 float64
Embarked_Q              bool
Embarked_S              bool
AgeGroup_Teenager       bool
AgeGroup_Adult          bool
AgeGroup_Senior         bool
dtype: object

In [63]:
from sklearn.model_selection import train_test_split

X = train.drop("Survived", axis=1)
y = train["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [64]:
X_train.shape, X_test.shape

((712, 11), (179, 11))

In [65]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)


In [66]:
from sklearn.metrics import accuracy_score

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(accuracy)

0.8156424581005587


In [67]:
test = pd.read_csv('/Users/wallace/Documents/Projects/titanic-end-to-end-model/data/test.csv')
test_ids = test['PassengerId']
test['Age'] = test['Age'].fillna(test.groupby('Pclass')['Age'].transform('median'))
test['Fare'] = test['Fare'].fillna(test['Fare'].median()) 
test['Embarked'] = test['Embarked'].fillna(test['Embarked'].mode()[0])

test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})

test['AgeGroup'] = pd.cut(test['Age'], bins=[0, 12, 18, 60, 100], labels=['Child', 'Teenager', 'Adult', 'Senior'])

test = pd.get_dummies(test, columns=['Embarked', 'AgeGroup'], drop_first=True)

In [68]:
X_test = test.drop(['PassengerId', 'Ticket', 'Name', 'Cabin'], axis=1)
X_test.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S,AgeGroup_Teenager,AgeGroup_Adult,AgeGroup_Senior
0,3,0,34.5,0,0,7.8292,True,False,False,True,False
1,3,1,47.0,1,0,7.0000,False,True,False,True,False
2,2,0,62.0,0,0,9.6875,True,False,False,False,True
3,3,0,27.0,0,0,8.6625,False,True,False,True,False
4,3,1,22.0,1,1,12.2875,False,True,False,True,False


In [69]:
test_predictions = model.predict(X_test)
submission = pd.DataFrame({"Survived" : test_predictions},
                          index=test["PassengerId"])

submission.to_csv("submission.csv", index=True)